# Analysis of US Tech Stock Returns and Risks
## 1. Introduction
This project conducts a comprehensive quantitative analysis of the return and risk characteristics of six leading U.S. technology stocks (Apple, Microsoft, Alphabet, Amazon, Tesla, Meta) from January 2021 to April 2025. The analysis covers core investment metrics including annualized return, annualized volatility, Sharpe ratio, maximum drawdown, and return correlation, providing data-driven investment insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['axes.unicode_minus'] = False

## 2. Data Setup

In [ ]:
np.random.seed(42)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META']
ticker_names = {
    'AAPL': 'Apple',
    'MSFT': 'Microsoft',
    'GOOGL': 'Alphabet',
    'AMZN': 'Amazon',
    'TSLA': 'Tesla',
    'META': 'Meta'
}

dates = pd.date_range(start='2021-01-01', end='2025-04-01', freq='B')

## 3. Simulated Data Generation (No Internet Needed)

In [ ]:
daily_returns = pd.DataFrame({
    'AAPL': np.random.normal(0.0008, 0.02, len(dates)),
    'MSFT': np.random.normal(0.0009, 0.018, len(dates)),
    'GOOGL': np.random.normal(0.0007, 0.019, len(dates)),
    'AMZN': np.random.normal(0.0006, 0.022, len(dates)),
    'TSLA': np.random.normal(0.0012, 0.035, len(dates)),
    'META': np.random.normal(0.0010, 0.025, len(dates))
}, index=dates)

data = pd.DataFrame()
for tick in tickers:
    data[tick] = 100 * (1 + daily_returns[tick]).cumprod()

print(f"Data shape: {data.shape}")

## 4. Risk & Return Performance Calculation

In [ ]:
annual_returns = daily_returns.mean() * 252 * 100
annual_volatility = daily_returns.std() * np.sqrt(252) * 100

risk_free_rate = 0.02
sharpe_ratio = (annual_returns/100 - risk_free_rate) / (annual_volatility/100)

max_drawdown = {}
for col in daily_returns.columns:
    cum_ret = (1 + daily_returns[col]).cumprod()
    run_max = cum_ret.cummax()
    dd = (cum_ret - run_max) / run_max
    max_drawdown[col] = dd.min() * 100

performance = pd.DataFrame({
    'Annual Return (%)': annual_returns,
    'Annual Volatility (%)': annual_volatility,
    'Sharpe Ratio': sharpe_ratio,
    'Max Drawdown (%)': max_drawdown
}).round(2)
performance['Company'] = performance.index.map(ticker_names)

print("\n===== Stock Performance Summary =====")
print(performance)

## 5. Cumulative Returns Visualization

In [ ]:
cum_returns = (1 + daily_returns).cumprod()

plt.figure(figsize=(12, 6))
for col in cum_returns.columns:
    plt.plot(cum_returns.index, cum_returns[col], linewidth=2, label=ticker_names[col])
plt.title('Cumulative Returns of US Tech Stocks (2021-2025)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Cumulative Return (Initial $1)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Risk-Return Scatter Plot

In [ ]:
plt.figure(figsize=(10, 6))
for t in tickers:
    plt.scatter(annual_volatility[t], annual_returns[t], s=150, alpha=0.8)
    plt.text(annual_volatility[t]+0.5, annual_returns[t]+0.5, ticker_names[t], fontsize=10)
plt.title('Risk-Return Profile of US Tech Stocks', fontsize=14)
plt.xlabel('Annualized Volatility (%)')
plt.ylabel('Annualized Return (%)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Return Correlation Heatmap

In [ ]:
corr = daily_returns.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
            xticklabels=[ticker_names[t] for t in corr.columns],
            yticklabels=[ticker_names[t] for t in corr.columns])
plt.title('Return Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Maximum Drawdown Visualization

In [ ]:
drawdown = (cum_returns - cum_returns.cummax()) / cum_returns.cummax()

plt.figure(figsize=(12, 6))
for col in drawdown.columns:
    plt.plot(drawdown.index, drawdown[col], linewidth=1.5, label=ticker_names[col])
plt.title('Maximum Drawdown of US Tech Stocks')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Conclusion & Investment Insights

In [ ]:
print("\n===== Investment Insights =====")
print(f"1. Best Risk-Adjusted Return (Highest Sharpe Ratio): {performance.sort_values('Sharpe Ratio', ascending=False).iloc[0]['Company']}")
print(f"2. Highest Annual Return: {performance.sort_values('Annual Return (%)', ascending=False).iloc[0]['Company']}")
print(f"3. Lowest Volatility (Lowest Risk): {performance.sort_values('Annual Volatility (%)').iloc[0]['Company']}")
print(f"4. Smallest Maximum Drawdown (Lowest Downside Risk): {performance.sort_values('Max Drawdown (%)', ascending=False).iloc[0]['Company']}")